# Stage 1: Experimental — VWAP-based wallet preselection

Preselect larger wallet sets that are profitable (average_roi > 0.02), then
compute per-trade trailing 15-minute VWAP and VWAP volume for BUY trades
by (wallet, condition_id, token_id).

These VWAP features are added back to the trade DataFrames for downstream
analysis.

**Output:** `stage1_experimental_result.json` with preselected wallets + VWAP stats.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from lib import (
    load_trades,
    compute_copyable_notional,
    compute_opening_metrics,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


## Parameters

In [2]:
MIN_ROI = 0.03
VWAP_WINDOW_MINUTES = 15

print(f"MIN_ROI: {MIN_ROI}")
print(f"VWAP_WINDOW_MINUTES: {VWAP_WINDOW_MINUTES}")

MIN_ROI: 0.03
VWAP_WINDOW_MINUTES: 15


## Load data

In [3]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)

train_cutoff = pd.Timestamp("2026-06-01", tz="UTC")
val_cutoff = pd.Timestamp("2026-07-01", tz="UTC")

df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Split by trade date:")
print(f"  Train: {len(df_train):>10,} trades  ({df_train['condition_id'].nunique():>5,} markets)  < {train_cutoff.date()}")
print(f"  Val:   {len(df_val):>10,} trades  ({df_val['condition_id'].nunique():>5,} markets)  {train_cutoff.date()} .. {val_cutoff.date()}")
print(f"  Test:  {len(df_test):>10,} trades  ({df_test['condition_id'].nunique():>5,} markets)  >= {val_cutoff.date()}")
print(f"  Total: {len(df_full):>10,} trades  ({df_full['condition_id'].nunique():>5,} markets)")

# Market overlap check
train_markets = set(df_train["condition_id"].unique())
val_markets = set(df_val["condition_id"].unique())
test_markets = set(df_test["condition_id"].unique())
print(f"\n  Markets overlapping train/val: {len(train_markets & val_markets)}")
print(f"  Markets overlapping train/test: {len(train_markets & test_markets)}")
print(f"  Markets overlapping val/test: {len(val_markets & test_markets)}")

Markets: 1974837


Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...


Total trades loaded: 14,250,603


Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00


Split by trade date:


  Train:  6,035,492 trades  (23,237 markets)  < 2026-06-01


  Val:    4,766,255 trades  (19,787 markets)  2026-06-01 .. 2026-07-01


  Test:   3,448,856 trades  (16,333 markets)  >= 2026-07-01


  Total: 14,250,603 trades  (56,875 markets)



  Markets overlapping train/val: 1350
  Markets overlapping train/test: 1
  Markets overlapping val/test: 1132


## Compute wallet metrics on training data

In [4]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3584


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.023563,NaN,12.845816,0.037440,53
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.005054,0.002003,-3.331733,0.000000,235
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.026175,0.036027,227.315103,0.011217,5511
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,NaN,0.000000,NaN,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.677602,-1.000000,119.386767,0.504015,50
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,NaN,-20.495392,NaN,344
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.012020,-1.000000,12.681270,-0.101257,81
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.025917,-0.181527,136.243576,0.005965,5369
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.196781,-0.063492,-316.535035,0.000000,1985
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.005820,-0.969739,24.241672,0.005206,2999


## Preselect wallets by average buy ROI

In [5]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

In [6]:
preselected_ws = set(
    wallet_vol.loc[wallet_vol["buy_roi"] > MIN_ROI, "wallet"]
)
print(f"Preselected wallets (buy_roi > {MIN_ROI}): {len(preselected_ws)}")

# Quick stats on the preselected set
preselected_df = wallet_vol[wallet_vol["wallet"].isin(preselected_ws)].copy()
print()
print(f"  Buy ROI range:  {preselected_df['buy_roi'].min():.4f} — {preselected_df['buy_roi'].max():.4f}")
print(f"  Avg num_buckets: {preselected_df['num_buckets'].mean():.0f}")
print(f"  total_pnl: ${preselected_df['total_pnl'].sum():,.0f}")
print(f"  trades: {preselected_df['trade_count'].sum():,.0f}")

Preselected wallets (buy_roi > 0.03): 1441

  Buy ROI range:  0.0304 — 199.0000
  Avg num_buckets: 1329
  total_pnl: $1,123,268
  trades: 2,164,403


In [7]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

## Compute 15-min trailing VWAP for BUY trades

For each BUY trade by a preselected wallet, look back 15 minutes over
the same (wallet, condition_id, token_id) and compute:
- **vwap_15m**: volume-weighted average price of trades **strictly before** this one
- **vwap_volume_15m**: total USDC volume of trades in the window

> `TEST_MODE=True` limits to 50 wallets for quick validation.

In [8]:
# buy_mask = df_full["wallet"].isin(preselected_ws) & (df_full["side"] == "BUY") 
# buy_trades = df_full.loc[buy_mask].copy() 
# print(f"BUY trades by preselected wallets: {len(buy_trades):,}")

# buy_trades = buy_trades.sort_values(
#     ["wallet", "condition_id", "token_id", "dt"],
#     kind="mergesort"
# ).reset_index(drop=True)

# buy_trades["vwap_15m"] = np.nan
# buy_trades["vwap_volume_15m"] = 0.0

# window_ns = np.timedelta64(VWAP_WINDOW_MINUTES, "m")

# for _, idx in buy_trades.groupby(
#     ["wallet", "condition_id", "token_id"],
#     sort=False
# ).groups.items():

#     g = buy_trades.loc[idx]

#     t = g["dt"].values
#     qty = g["quantity"].to_numpy()
#     usdc = g["usdc_amount"].to_numpy()
#     pq = (g["price"] * g["quantity"]).to_numpy()

#     left = 0
#     sum_qty = 0.0
#     sum_pq = 0.0
#     sum_usdc = 0.0

#     out_vwap = np.empty(len(g))
#     out_vol = np.empty(len(g))

#     for i in range(len(g)):

#         # remove expired trades
#         while left < i and t[left] < t[i] - window_ns:
#             sum_qty -= qty[left]
#             sum_pq -= pq[left]
#             sum_usdc -= usdc[left]
#             left += 1

#         # current trade is excluded
#         out_vwap[i] = np.nan if sum_qty == 0 else sum_pq / sum_qty
#         out_vol[i] = sum_usdc

#         # add current trade
#         sum_qty += qty[i]
#         sum_pq += pq[i]
#         sum_usdc += usdc[i]

#     buy_trades.loc[idx, "vwap_15m"] = out_vwap
#     buy_trades.loc[idx, "vwap_volume_15m"] = out_vol

# df_full = df_full.merge(
#     buy_trades[["tx_hash","vwap_15m","vwap_volume_15m"]],
#     on="tx_hash",
#     how="left",
# )

# df_train = df_full[df_full["dt"] < train_cutoff].copy()
# df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
# df_test = df_full[df_full["dt"] >= val_cutoff].copy()

In [9]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

In [10]:
bad_buy_leaders = wallet_vol[
    (wallet_vol['buy_roi'] >= 0.04)
    & (wallet_vol['trade_count'] >= 1000)
    & (wallet_vol['total_pnl'] > 10000)
    & (wallet_vol['max_drawdown_to_pnl'] <= 0.5)
    & (wallet_vol['copyable_roi'] < 0.02)
    & (wallet_vol['buy_copyable_pnl'] * -1 >= 1000)
]

print(f"Bad buy leaders: {len(bad_buy_leaders)}")
print(f"Bad buy leader train trades: {len(df_train[df_train['wallet'].isin(bad_buy_leaders['wallet'])])}")
print(f"Bad buy leader pnl: ${bad_buy_leaders['total_pnl'].sum():,.0f}")
print(f"Bad buy leader copyable buy pnl: ${bad_buy_leaders['buy_copyable_pnl'].sum():,.0f}")
print(f"Bad buy leader buy roi: ${bad_buy_leaders['buy_pnl'].sum() / bad_buy_leaders['buy_notional'].sum():,.2f}")
print(f"Bad buy leader val pnl: ${df_val[df_val['wallet'].isin(bad_buy_leaders['wallet'])]['pnl'].sum():,.0f}")

Bad buy leaders: 2
Bad buy leader train trades: 133447
Bad buy leader pnl: $29,390
Bad buy leader copyable buy pnl: $-11,750
Bad buy leader buy roi: $0.07


Bad buy leader val pnl: $9,746


In [11]:
bad_buy_leaders.head()

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,return,copyable_pnl_factor,copyable_roi,opening_pnl,opening_notional,opening_copyable_pnl,opening_buys,opening_copyable_notional,opening_roi,opening_copyable_roi
1153,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0.322946,92384,10339,103599,263354.778806,15099.549014,-9365.720930,0.275840,0.435316,...,0.057335,0.0,0.0,1351.187818,17103.279037,-271.397678,10298.0,6479.346074,0.079002,-0.041887
1289,0xc7d02944a76b9f83b199e9090ecc92c82d241f8a,0.434007,28639,4309,29848,165292.295427,14290.413849,-3584.065829,0.300944,0.409550,...,0.086455,0.0,0.0,3230.185197,43866.304970,-1175.765783,5857.0,10039.604894,0.073637,-0.117113


In [12]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='object')

In [13]:
(
    wallet_vol[
        (wallet_vol['trade_count'] >= 500)
        # & (wallet_vol['average_roi'] >= 0.05)
        # & (wallet_vol['copyable_pnl'] >= 100)
        ]
    ).sort_values('total_pnl', ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,return,copyable_pnl_factor,copyable_roi,opening_pnl,opening_notional,opening_copyable_pnl,opening_buys,opening_copyable_notional,opening_roi,opening_copyable_roi
1149,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,0.957756,3163,394,3718,1.092031e+05,30043.942187,11841.913827,1.032128,1.162751,...,0.275120,0.394153,0.487646,3969.194801,6349.977287,513.202781,446.0,1857.731022,0.625072,0.276252
1143,0xae2e04fe9d8ccba5e45ba17ddf9dfbef498c40ad,1.857969,3859,484,4716,1.033217e+05,22667.525475,2248.495040,1.158076,1.258107,...,0.219388,0.099195,0.065497,4883.986659,10320.547633,-913.980441,580.0,3223.326167,0.473229,-0.283552
1173,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0.097495,266467,7853,286565,1.211482e+06,18046.306392,-23867.685176,0.106982,0.180791,...,0.014896,0.000000,0.000000,2189.775995,210062.939829,-3181.590138,46879.0,82367.297536,0.010424,-0.038627
498,0x488c725253fc21c7a9ca812030dc2f6343f98c1c,1.609136,2359,391,3269,1.861033e+05,16867.802367,2569.051445,0.732761,0.990217,...,0.090637,0.152305,0.089965,8224.460232,25504.823979,371.075386,475.0,3683.864651,0.322467,0.100730
1153,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0.322946,92384,10339,103599,2.633548e+05,15099.549014,-9365.720930,0.275840,0.435316,...,0.057335,0.000000,0.000000,1351.187818,17103.279037,-271.397678,10298.0,6479.346074,0.079002,-0.041887
1289,0xc7d02944a76b9f83b199e9090ecc92c82d241f8a,0.434007,28639,4309,29848,1.652923e+05,14290.413849,-3584.065829,0.300944,0.409550,...,0.086455,0.000000,0.000000,3230.185197,43866.304970,-1175.765783,5857.0,10039.604894,0.073637,-0.117113
947,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,0.786399,3379,358,4248,8.515574e+04,12334.921901,3913.186756,0.431021,0.664747,...,0.144851,0.317245,0.035503,2281.637325,12308.219513,1063.131160,413.0,3591.994193,0.185375,0.295972
927,0x8d0930676d559cc8fb7d8af0c555791c1820143f,0.112830,5356,697,5896,1.150638e+06,11769.118661,2002.084483,0.261282,0.403760,...,0.010228,0.170113,0.089243,4130.966471,245683.787199,849.066007,711.0,10868.312838,0.016814,0.078123
1585,0xf1e18ec32b2f1e123bc098e3956e6fd00012c152,1.729986,1437,155,2745,3.164178e+04,11222.055485,2332.129447,0.696916,0.872260,...,0.354659,0.207817,0.436024,1971.908374,2595.435486,436.082675,166.0,640.238506,0.759760,0.681125
1410,0xdafdc201e4a769c4424462f9210763b221da2ad4,2.106591,1691,278,2174,9.428722e+04,10559.840084,943.980646,1.078021,1.280717,...,0.111997,0.089393,0.004235,2816.186397,27036.603046,856.983938,290.0,3244.420900,0.104162,0.264141


In [14]:
if 'bad_leader_wallet' not in df_full.columns:
    bad_buy_leader_trades = df_full[(df_full['wallet'].isin(bad_buy_leaders['wallet'])) & (df_full['side'] == 'BUY')]
    print(f"Bad buy leader full trades: {len(bad_buy_leader_trades)}")

    leaders = bad_buy_leader_trades.rename(columns={"dt": "dt_leader", "wallet": "bad_leader_wallet"})[['dt_leader', 'bad_leader_wallet', 'condition_id', 'outcome']]

    df_full = pd.merge_asof(
        df_full.sort_values("dt"),
        leaders.sort_values("dt_leader"),
        left_on="dt",
        right_on="dt_leader",
        by=["condition_id", "outcome"],
        direction="backward",
        tolerance=pd.Timedelta(minutes=5),
        allow_exact_matches=False,
)

Bad buy leader full trades: 171233


In [15]:
df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Bad leader trades in train: {len(df_train[df_train['bad_leader_wallet'].notnull()])}")
print(f"Bad leader trades in val: {len(df_val[df_val['bad_leader_wallet'].notnull()])}")
print(f"Bad leader trades in test: {len(df_test[df_test['bad_leader_wallet'].notnull()])}")

Bad leader trades in train: 310208
Bad leader trades in val: 114080
Bad leader trades in test: 53493


## Signal Quality Framework

We evaluate each signal using **Information Coefficient (IC)** and
**Information Ratio (IR)**, following Grinold & Kahn (1999),
*Active Portfolio Management* (McGraw-Hill).

| Metric | Definition | Interpretation |
|--------|------------|----------------|
| **IC** | Spearman rank correlation between signal value and forward copyable ROI | Does a higher signal predict better PnL? |
| **IR** | Mean(IC) / Std(IC) across daily chunks | How consistent is the predictive power? |
| **Hit Rate** | % of events where signal sign matches PnL sign | Directional accuracy |
| **Bootstrap CI** | 2.5th-97.5th percentile of IC over 10k resamples | Is IC sign reliably non-zero? |

Signal overlap is measured with **coincidence rate** (do they fire together?)
and **IC correlation** (are their predictions redundant?).

In [16]:

# Signal quality: IC, IR, bootstrap, overlap, combination

import numpy as np
import pandas as pd


def _rankdata(v):
    """Fractional ranking (scipy.stats.rankdata, method='average')."""
    n = len(v)
    sorter = np.argsort(v, kind="mergesort")
    ordinal = np.empty(n, dtype=np.intp)
    ordinal[sorter] = np.arange(n)
    inv = np.argsort(sorter, kind="mergesort")
    rank = ordinal + 1.0
    i = 0
    while i < n:
        j = i + 1
        while j < n and v[sorter[j]] == v[sorter[i]]:
            j += 1
        if j > i + 1:
            avg_rank = (i + j + 1) / 2.0
            for k in range(i, j):
                rank[sorter[k]] = avg_rank
        i = j
    return rank[inv]


def spearman_rho(x, y):
    """Spearman rank correlation (numpy-only)."""
    mask = np.isfinite(x) & np.isfinite(y)
    n = mask.sum()
    if n < 10:
        return np.nan
    rx = _rankdata(x[mask].values if hasattr(x, 'values') else x[mask])
    ry = _rankdata(y[mask].values if hasattr(y, 'values') else y[mask])
    rx_m = rx.mean()
    ry_m = ry.mean()
    num = np.sum((rx - rx_m) * (ry - ry_m))
    den = np.sqrt(np.sum((rx - rx_m)**2) * np.sum((ry - ry_m)**2))
    return num / den if den != 0 else np.nan


def compute_event_ic(signal, forward_roi):
    """IC: rank correlation between signal and forward copyable ROI."""
    return spearman_rho(signal, forward_roi)


def compute_event_ir(signal, forward_roi, timestamps, freq="D"):
    """IR = mean(IC_chunk) / std(IC_chunk) across time chunks.
    
    Higher IR means predictive power is consistent (Grinold & Kahn Ch. 7).
    """
    ts = timestamps
    chunks = pd.Series(index=pd.DatetimeIndex(ts), data=np.arange(len(signal))).groupby(
        pd.Grouper(freq=freq)
    )
    ics = []
    for _, idx in chunks:
        if len(idx) < 5:
            continue
        rho = compute_event_ic(signal.iloc[idx], forward_roi.iloc[idx])
        if not np.isnan(rho):
            ics.append(rho)
    if len(ics) < 3:
        return np.nan
    arr = np.array(ics)
    return float(arr.mean() / arr.std(ddof=1)) if arr.std(ddof=1) > 0 else np.nan


def bootstrap_ic(signal, forward_roi, n_iter=10_000, alpha=0.05, seed=42):
    """Bootstrap CI for IC (Efron & Tibshirani 1993).
    
    Returns (mean_ic, ci_lower, ci_upper).
    """
    mask = signal.notna() & forward_roi.notna()
    s = signal[mask].values
    p = forward_roi[mask].values
    n = len(s)
    if n < 10:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_ics = np.empty(n_iter)
    for i in range(n_iter):
        idx = rng.integers(0, n, n)
        boot_ics[i] = spearman_rho(s[idx], p[idx])
    mean_ic = float(np.nanmean(boot_ics))
    ci_lo = float(np.nanpercentile(boot_ics, 100 * alpha / 2))
    ci_hi = float(np.nanpercentile(boot_ics, 100 * (1 - alpha / 2)))
    return mean_ic, ci_lo, ci_hi


def hit_rate(signal, forward_roi):
    """Fraction of events where signal sign matches PnL sign."""
    mask = signal.notna() & forward_roi.notna() & (forward_roi != 0)
    if mask.sum() < 10:
        return np.nan
    sgn_sig = np.sign(signal[mask])
    sgn_pnl = np.sign(forward_roi[mask])
    return float((sgn_sig == sgn_pnl).mean())


def signal_quality_report(signals_df, signal_cols, roi_col="copyable_roi",
                           dt_col="dt", ir_freq="D", n_bootstrap=5_000):
    """Compute IC, IR, hit rate, and bootstrap CI for each signal.
    
    Returns DataFrame with one row per signal.
    """
    rows = []
    for col in signal_cols:
        ic = compute_event_ic(signals_df[col], signals_df[roi_col])
        ir = compute_event_ir(signals_df[col], signals_df[roi_col],
                               signals_df[dt_col], freq=ir_freq)
        hr = hit_rate(signals_df[col], signals_df[roi_col])
        m, clo, chi = bootstrap_ic(signals_df[col], signals_df[roi_col],
                                    n_iter=n_bootstrap)
        rows.append({
            "signal": col,
            "IC": ic,
            "IR": ir,
            "hit_rate": hr,
            "bootstrap_mean_ic": m,
            "bootstrap_ci_lo": clo,
            "bootstrap_ci_hi": chi,
            "n_events": int(signals_df[col].notna().sum()),
        })
    return pd.DataFrame(rows).sort_values("IC", ascending=False, key=abs)


# === Signal overlap ===

def coincidence_rate(s1, s2):
    """Jaccard-like coincidence: P(both non-zero | either non-zero)."""
    both = ((s1.notna() & (s1 != 0)) & (s2.notna() & (s2 != 0))).sum()
    either = ((s1.notna() & (s1 != 0)) | (s2.notna() & (s2 != 0))).sum()
    return both / either if either > 0 else 0.0


def ic_correlation_matrix(signals_df, signal_cols, roi_col="copyable_roi"):
    """Pairwise IC of signal values on overlapping events."""
    n = len(signal_cols)
    mat = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            if i == j:
                mat[i, j] = 1.0
                continue
            both = signals_df[signal_cols[i]].notna() & signals_df[signal_cols[j]].notna()
            if both.sum() < 10:
                continue
            mat[i, j] = compute_event_ic(
                signals_df.loc[both, signal_cols[i]],
                signals_df.loc[both, signal_cols[j]],
            )
    return pd.DataFrame(mat, index=signal_cols, columns=signal_cols)


# === Signal combination ===

def compute_optimal_weights(
    signals_df, signal_cols, roi_col="copyable_roi",
    shrinkage=0.5,
):
    """Markowitz-optimal signal weights with shrinkage (Grinold & Kahn Ch. 13).
    
    w = (1-lambda) * inv(Sigma) * IC + lambda * (1/n)
    
    Parameters
    ----------
    shrinkage : float
        0 = full Markowitz, 1 = equal weight.
    
    Returns
    -------
    pd.Series of weights indexed by signal_cols.
    """
    n = len(signal_cols)
    ic_vec = np.array([
        compute_event_ic(signals_df[c], signals_df[roi_col]) or 0.0
        for c in signal_cols
    ])
    
    valid = signals_df[signal_cols].notna().all(axis=1)
    if valid.sum() < 10 or n <= 1:
        return pd.Series(np.ones(n) / n, index=signal_cols)
    
    sig_vals = signals_df.loc[valid, signal_cols].values
    cov = np.cov(sig_vals, rowvar=False)
    avg_var = np.trace(cov) / n
    shrunk_cov = (1 - shrinkage) * cov + shrinkage * np.eye(n) * avg_var
    
    try:
        inv_cov = np.linalg.solve(shrunk_cov, np.eye(n))
        w = inv_cov @ ic_vec
        w_abs_sum = np.sum(np.abs(w))
        if w_abs_sum > 1e-12:
            w = w / w_abs_sum
        else:
            w = np.ones(n) / n
    except np.linalg.LinAlgError:
        w = np.ones(n) / n
    
    return pd.Series(w, index=signal_cols)


def apply_composite_score(signals_df, signal_cols, weights):
    """Composite signal = sum w_i * signal_i."""
    result = np.zeros(len(signals_df))
    for col in signal_cols:
        result += weights[col] * signals_df[col].fillna(0.0).values
    return pd.Series(result, index=signals_df.index)


def cs_rank(s, grouper=None):
    """Cross-sectional rank transform. Maps values to [-1, 1] within groups.
    
    If grouper is provided, ranks within each group independently.
    Standard Grinold & Kahn normalization.
    """
    if grouper is not None:
        result = s.groupby(grouper, sort=False).transform(
            lambda x: 2.0 * (_rankdata(x.values) - 1.0) / max(len(x) - 1, 1) - 1.0
        )
    else:
        n = len(s)
        r = _rankdata(s.values) if hasattr(s, 'values') else _rankdata(np.asarray(s))
        result = 2.0 * (r - 1.0) / max(n - 1, 1) - 1.0
    return result


## Parameters & Test Mode

In [17]:

# Test mode
TEST_MODE = False
MAX_CANDIDATE_WALLETS = 20
MAX_CANDIDATE_TRADES = 5000

# Signal windows (minutes)
BAD_LEADER_WINDOW = 5
QUALITY_WALLET_WINDOW = 15
VWAP_WINDOW = 15

# Quality wallet definition
QW_MIN_BUY_ROI = 0.05
QW_MIN_BUCKETS = 50
QW_MAX_DD_TO_PNL = 0.3
QW_MIN_COPYABLE_ROI = 0.02

# Copy universe
COPY_MIN_BUY_ROI = 0.03
COPY_MIN_BUCKETS = 20
COPY_MIN_MARKETS = 15
COPY_MAX_DD_TO_PNL = 0.2
COPY_MIN_COPYABLE_ROI = 0.05

print(f"TEST_MODE: {TEST_MODE}")
print(f"Signals: bad_leader ({BAD_LEADER_WINDOW}min), "
      f"quality_wallet ({QUALITY_WALLET_WINDOW}min), "
      f"vwap_deviation ({VWAP_WINDOW}min)")


TEST_MODE: False
Signals: bad_leader (5min), quality_wallet (15min), vwap_deviation (15min)


## Define Copy Universe (candidate trades)

Trades we *could* copy. BUY trades by wallets that pass a quality filter.

In [18]:

# Copy universe: wallets that pass quality + stability filter
copy_mask = (
    (wallet_vol['buy_roi'] >= COPY_MIN_BUY_ROI)
    & (wallet_vol['num_buckets'] >= COPY_MIN_BUCKETS)
    & (wallet_vol['num_markets'] >= COPY_MIN_MARKETS)
    & (wallet_vol['max_drawdown_to_pnl'].fillna(1.0) <= COPY_MAX_DD_TO_PNL)
    & (wallet_vol['copyable_roi'].fillna(0.0) >= COPY_MIN_COPYABLE_ROI)
)
copy_wallets = set(wallet_vol.loc[copy_mask, 'wallet'])
print(f"Copy universe: {len(copy_wallets)} wallets")

if TEST_MODE and len(copy_wallets) > MAX_CANDIDATE_WALLETS:
    copy_wallets = set(
        wallet_vol.loc[copy_mask].sort_values('buy_roi', ascending=False)
        .head(MAX_CANDIDATE_WALLETS)['wallet']
    )
    print(f"  (test mode: {len(copy_wallets)} wallets)")

# BUY trades by copy-universe wallets
candidate_mask = df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
candidate_trades = df_full[candidate_mask].copy()
print(f"Candidate BUY trades: {len(candidate_trades):,}")

if TEST_MODE and len(candidate_trades) > MAX_CANDIDATE_TRADES:
    candidate_trades = candidate_trades.sample(MAX_CANDIDATE_TRADES, random_state=42)
    print(f"  (test mode: sampled {len(candidate_trades)})")

c_train = candidate_trades[candidate_trades['dt'] < train_cutoff].copy()
c_val = candidate_trades[
    (candidate_trades['dt'] >= train_cutoff) & (candidate_trades['dt'] < val_cutoff)
].copy()
c_test = candidate_trades[candidate_trades['dt'] >= val_cutoff].copy()
print(f"  Train: {len(c_train):,}  Val: {len(c_val):,}  Test: {len(c_test):,}")


Copy universe: 55 wallets


Candidate BUY trades: 215,023
  Train: 70,868  Val: 82,959  Test: 61,196


## Signal 1: Bad Leader Proximity

**Hypothesis:** Trades on the same (condition_id, outcome) recently after a
"bad leader" (profitable but uncopyable wallet) tend to underperform.

Already computed via `merge_asof` above. Column: `bad_leader_wallet`.

In [19]:

# Signal 1: bad_leader_buy (binary)
# Already computed in earlier cells (bad_leader_wallet column on df_full)

c_train['sig_bad_leader'] = c_train['bad_leader_wallet'].notna().astype(float)
c_val['sig_bad_leader'] = c_val['bad_leader_wallet'].notna().astype(float)
c_test['sig_bad_leader'] = c_test['bad_leader_wallet'].notna().astype(float)

print("Signal 1: bad_leader_buy (binary)")
for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
    rate = df_c['sig_bad_leader'].mean()
    ic_v = compute_event_ic(df_c['sig_bad_leader'], df_c['copyable_roi'])
    print(f"  {label}: firing_rate={rate:.4f}  IC={ic_v:.4f}")


Signal 1: bad_leader_buy (binary)
  Train: firing_rate=0.0711  IC=-0.0009
  Val: firing_rate=0.0299  IC=0.0005
  Test: firing_rate=0.0167  IC=0.0040


## Signal 2: Quality Wallet Proximity

**Hypothesis:** When high-quality wallets buy a token, copying within a
window yields positive PnL. Quality wallets have high buy_roi, diversification,
and stable returns.

In [20]:

# Signal 2: quality_wallet_proximity

# Define quality wallets from training-period metrics
qw_mask = (
    (wallet_vol['buy_roi'] >= QW_MIN_BUY_ROI)
    & (wallet_vol['num_buckets'] >= QW_MIN_BUCKETS)
    & (wallet_vol['max_drawdown_to_pnl'].fillna(1.0) <= QW_MAX_DD_TO_PNL)
    & (wallet_vol['copyable_roi'].fillna(0.0) >= QW_MIN_COPYABLE_ROI)
)
quality_wallets = set(wallet_vol.loc[qw_mask, 'wallet'])
print(f"Quality wallets: {len(quality_wallets)}")

# merge_asof: for each trade, find nearest quality-wallet buy on same
# (condition_id, outcome) in last QUALITY_WALLET_WINDOW minutes
# NOTE: old candidate slices (c_train/c_val/c_test) do NOT have this column yet;
# we will re-slice them from df_full in the next cell.
if 'qw_wallet' not in df_full.columns:
    qw_buys = df_full[
        df_full['wallet'].isin(quality_wallets) & (df_full['side'] == 'BUY')
    ].copy()
    qw_buys = qw_buys.rename(columns={
        'dt': 'dt_qw', 'wallet': 'qw_wallet', 'usdc_amount': 'qw_usdc'
    })[['dt_qw', 'qw_wallet', 'qw_usdc', 'condition_id', 'outcome']].sort_values('dt_qw')
    print(f"  Quality wallet BUY trades: {len(qw_buys):,}")

    df_full = pd.merge_asof(
        df_full.sort_values('dt'),
        qw_buys,
        left_on='dt', right_on='dt_qw',
        by=['condition_id', 'outcome'],
        direction='backward',
        tolerance=pd.Timedelta(minutes=QUALITY_WALLET_WINDOW),
        allow_exact_matches=False,
        suffixes=('', '_qw'),
    )
    print("  Signal 2 column 'qw_wallet' added to df_full")
else:
    print("  Signal 2 already computed, skipping merge_asof")

# Candidate sets will be refreshed from df_full in the next cell,
# at which point they will have qw_wallet + qw_usdc columns.
print("  (candidate trades will be re-sliced next)")


Quality wallets: 103


  Quality wallet BUY trades: 307,018


  Signal 2 column 'qw_wallet' added to df_full
  (candidate trades will be re-sliced next)


## Refresh candidate trades

Re-slice candidate trades from the fully-annotated `df_full` (now includes quality_wallet signal).

In [21]:
# Re-slice candidate trades after signal columns added to df_full
candidate_mask = df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
candidate_trades = df_full[candidate_mask].copy()
print(f"Candidate BUY trades (refreshed): {len(candidate_trades):,}")

if TEST_MODE and len(candidate_trades) > MAX_CANDIDATE_TRADES:
    candidate_trades = candidate_trades.sample(MAX_CANDIDATE_TRADES, random_state=42)
    print(f"  (test mode: sampled {len(candidate_trades)})")

c_train = candidate_trades[candidate_trades['dt'] < train_cutoff].copy()
c_val = candidate_trades[
    (candidate_trades['dt'] >= train_cutoff) & (candidate_trades['dt'] < val_cutoff)
].copy()
c_test = candidate_trades[candidate_trades['dt'] >= val_cutoff].copy()
print(f"  Train: {len(c_train):,}  Val: {len(c_val):,}  Test: {len(c_test):,}")


Candidate BUY trades (refreshed): 215,023
  Train: 70,868  Val: 82,959  Test: 61,196


## Assign signal columns

Derive signal columns from `df_full` columns on refreshed candidate sets.

In [22]:
# Assign signal columns on refreshed candidate sets
# (df_full already has bad_leader_wallet and qw_wallet columns)
for df_c in [c_train, c_val, c_test]:
    # Signal 1: bad leader proximity
    df_c['sig_bad_leader'] = df_c['bad_leader_wallet'].notna().astype(float)
    # Signal 2: quality wallet proximity
    df_c['sig_qw_any'] = df_c['qw_wallet'].notna().astype(float)
    df_c['sig_qw_volume'] = df_c['qw_usdc'].fillna(0.0)

# Print IC for each signal
for sig in ['sig_bad_leader', 'sig_qw_any', 'sig_qw_volume']:
    print(f"{sig}:")
    for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
        rate = df_c[sig].mean() if df_c[sig].dtype.kind in 'bif' else df_c[sig].notna().mean()
        ic_v = compute_event_ic(df_c[sig], df_c['copyable_roi'])
        print(f"  {label}: firing_rate={rate:.4f}  IC={ic_v:.4f}")
    print()
for sig in ['sig_vwap_csrank', 'sig_vwap_signed']:
    if sig in c_train.columns:
        print(f"{sig}:")
        for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
            ic_cr = compute_event_ic(df_c[sig], df_c['copyable_roi'])
            ic_pnl = compute_event_ic(df_c[sig], df_c['pnl'])
            print(f"  {label}: IC(copyable_roi)={ic_cr:.4f}  IC(pnl)={ic_pnl:.4f}")
        print()


sig_bad_leader:
  Train: firing_rate=0.0711  IC=0.0005
  Val: firing_rate=0.0299  IC=-0.0051
  Test: firing_rate=0.0167  IC=0.0005

sig_qw_any:
  Train: firing_rate=0.5454  IC=-0.0066


  Val: firing_rate=0.5130  IC=-0.0043
  Test: firing_rate=0.4807  IC=0.0026

sig_qw_volume:
  Train: firing_rate=4.3585  IC=-0.0046


  Val: firing_rate=5.7784  IC=-0.0048
  Test: firing_rate=8.2325  IC=0.0072



## Signal 3: VWAP Deviation (simplified)

**Hypothesis:** Buying below the trailing VWAP of quality buyers is a better entry.
VWAP deviation = (price / vwap_15m) - 1.

For TEST_MODE: bucketed 5-min VWAP approximation.

In [23]:

# Signal 3: VWAP deviation (simplified for TEST_MODE)

VWAP_BUCKET_MINUTES = 5

if TEST_MODE:
    print("TEST_MODE: bucketed VWAP approximation")

    def floor_dt(s, freq=f"{VWAP_BUCKET_MINUTES}min"):
        return s.dt.floor(freq)

    # All BUY trades by copy-universe wallets
    copy_buys = df_full[
        df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
    ].copy()
    copy_buys['dt_bucket'] = floor_dt(copy_buys['dt'])

    # Per (condition_id, outcome, bucket): VWAP
    bucket_vwap = copy_buys.groupby(
        ['condition_id', 'outcome', 'dt_bucket'], sort=False
    ).apply(
        lambda g: pd.Series({
            'vwap': (g['price'] * g['quantity']).sum() / g['quantity'].sum(),
            'vwap_vol': g['usdc_amount'].sum(),
        }), include_groups=False
    ).reset_index()

    # Shift VWAP one bucket forward to avoid look-ahead
    bucket_vwap['dt_bucket_prev'] = bucket_vwap['dt_bucket'] - pd.Timedelta(minutes=VWAP_BUCKET_MINUTES)

    # For each candidate trade, merge on previous bucket
    _vwap_dfs = []
    for name, df_c in [('train', c_train), ('val', c_val), ('test', c_test)]:
        df_c['dt_bucket'] = floor_dt(df_c['dt'])
        df_c = df_c.merge(
            bucket_vwap,
            left_on=['condition_id', 'outcome', 'dt_bucket'],
            right_on=['condition_id', 'outcome', 'dt_bucket_prev'],
            how='left',
            suffixes=('', '_vwap'),
        )
        df_c['sig_vwap_dev'] = np.where(
            df_c['vwap'].notna() & (df_c['vwap'] > 0),
            (df_c['price'] / df_c['vwap']) - 1.0,
            np.nan,
        )
        # Signed: negative z-score = buying below VWAP = good entry
        df_c['sig_vwap_signed'] = np.where(
            df_c['sig_vwap_dev'].notna(),
            -df_c['sig_vwap_dev'],
            np.nan,
        )
        # Also keep magnitude for comparison
        df_c['sig_vwap_strength'] = df_c['sig_vwap_dev'].abs().fillna(0.0)
        _vwap_dfs.append((name, df_c))
    for name, df_c in _vwap_dfs:
        if name == 'train': c_train = df_c
        elif name == 'val': c_val = df_c
        elif name == 'test': c_test = df_c

else:
    print("Non-TEST_MODE: vectorized bucketed VWAP (previous bucket)")

    copy_buys = df_full[
        df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
    ].copy()
    copy_buys['dt_bucket'] = copy_buys['dt'].dt.floor(f"{VWAP_BUCKET_MINUTES}min")

    copy_buys['price_vol'] = copy_buys['price'] * copy_buys['quantity']
    bucket_vwap = copy_buys.groupby(
        ['condition_id', 'outcome', 'dt_bucket'], sort=False, observed=True
    ).agg(
        vwap_price=('price_vol', 'sum'),
        total_qty=('quantity', 'sum'),
        vwap_vol=('usdc_amount', 'sum'),
    ).reset_index()
    bucket_vwap['vwap'] = bucket_vwap['vwap_price'] / bucket_vwap['total_qty']

    bucket_vwap['dt_bucket_prev'] = bucket_vwap['dt_bucket'] - pd.Timedelta(minutes=VWAP_BUCKET_MINUTES)

    _vwap_dfs = []
    for name, df_c in [('train', c_train), ('val', c_val), ('test', c_test)]:
        df_c['dt_bucket'] = df_c['dt'].dt.floor(f"{VWAP_BUCKET_MINUTES}min")
        df_c = df_c.merge(
            bucket_vwap[['condition_id', 'outcome', 'dt_bucket_prev', 'vwap', 'vwap_vol']],
            left_on=['condition_id', 'outcome', 'dt_bucket'],
            right_on=['condition_id', 'outcome', 'dt_bucket_prev'],
            how='left',
            suffixes=('', '_vwap'),
        )
        df_c['sig_vwap_dev'] = np.where(
            df_c['vwap'].notna() & (df_c['vwap'] > 0),
            (df_c['price'] / df_c['vwap']) - 1.0,
            np.nan,
        )
        df_c['sig_vwap_signed'] = np.where(
            df_c['sig_vwap_dev'].notna(),
            -df_c['sig_vwap_dev'],
            np.nan,
        )
        df_c['sig_vwap_strength'] = df_c['sig_vwap_dev'].abs().fillna(0.0)
        _vwap_dfs.append((name, df_c))
    for name, df_c in _vwap_dfs:
        if name == 'train': c_train = df_c
        elif name == 'val': c_val = df_c
        elif name == 'test': c_test = df_c

# CS-rank VWAP signed deviation for comparability across days
for df_c in [c_train, c_val, c_test]:
    df_c['sig_vwap_csrank'] = cs_rank(df_c['sig_vwap_signed'].fillna(0.0), df_c['dt'].dt.date)

print()
for label, df_c in [("Train", c_train), ("Val", c_val), ("Test", c_test)]:
    cov = df_c['sig_vwap_dev'].notna().mean()
    ic_signed = compute_event_ic(df_c['sig_vwap_signed'], df_c['copyable_roi'])
    ic_strength = compute_event_ic(df_c['sig_vwap_strength'], df_c['copyable_roi'])
    ic_csrank = compute_event_ic(df_c['sig_vwap_csrank'], df_c['copyable_roi'])
    print(f"  {label}: coverage={cov:.3f}  IC(signed)={ic_signed:.4f}  IC(strength)={ic_strength:.4f}  IC(csrank)={ic_csrank:.4f}")


Non-TEST_MODE: vectorized bucketed VWAP (previous bucket)



  Train: coverage=0.258  IC(signed)=0.0096  IC(strength)=0.0049  IC(csrank)=-0.0103
  Val: coverage=0.219  IC(signed)=-0.0196  IC(strength)=0.0035  IC(csrank)=0.0134
  Test: coverage=0.202  IC(signed)=-0.0178  IC(strength)=-0.0084  IC(csrank)=-0.0276


## Signal Quality Report

Compute IC, IR, hit rate, and bootstrap confidence intervals on validation.

In [24]:

# Signal quality report on validation set
signal_cols = ['sig_bad_leader', 'sig_qw_any', 'sig_vwap_signed', 'sig_vwap_strength', 'sig_vwap_csrank']
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]

quality_report = signal_quality_report(
    c_val, active_cols,
    roi_col='copyable_roi', dt_col='dt',
    ir_freq='D', n_bootstrap=5_000,
)

print("Signal quality report (validation set):")
display(quality_report.round(4))


Signal quality report (validation set):


,signal,IC,IR,hit_rate,bootstrap_mean_ic,bootstrap_ci_lo,bootstrap_ci_hi,n_events
2,sig_vwap_signed,-0.0196,0.0126,0.4436,0.0002,-0.0210,0.0212,18181
4,sig_vwap_csrank,0.0134,-0.0828,0.5992,-0.0001,-0.0107,0.0102,82959
0,sig_bad_leader,-0.0051,0.0948,0.0141,0.0000,-0.0103,0.0103,82959
1,sig_qw_any,-0.0043,0.0878,0.1485,0.0002,-0.0101,0.0104,82959
3,sig_vwap_strength,0.0035,0.1954,0.0605,0.0000,-0.0100,0.0103,82959


## Signal Overlap Analysis

How redundant are the signals? Do they fire on the same events?

In [25]:

# Overlap analysis
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]
n_sig = len(active_cols)

if n_sig < 2:
    print("Need at least 2 active signals for overlap analysis")
else:
    # 1. Coincidence rate
    print("1. Coincidence rate (P(both fire | either fires)):")
    coin_mat = np.full((n_sig, n_sig), np.nan)
    for i, s1 in enumerate(active_cols):
        for j, s2 in enumerate(active_cols):
            coin_mat[i, j] = 1.0 if i == j else coincidence_rate(c_val[s1], c_val[s2])
    coin_df = pd.DataFrame(coin_mat, index=active_cols, columns=active_cols)
    display(coin_df.round(3))

    # 2. IC correlation
    print("\n2. IC correlation (signal value correlation):")
    ic_corr = ic_correlation_matrix(c_val, active_cols)
    display(ic_corr.round(3))

    # 3. Conditional IC
    print("\n3. Conditional IC (unique contribution):")
    for s in active_cols:
        other = [c for c in active_cols if c != s]
        neutral = np.ones(len(c_val), dtype=bool)
        for o in other:
            neutral &= (c_val[o].abs() < 0.01) | c_val[o].isna()
        if neutral.sum() < 20:
            continue
        ic_cond = compute_event_ic(c_val.loc[neutral, s], c_val.loc[neutral, 'copyable_roi'])
        ic_full = compute_event_ic(c_val[s], c_val['copyable_roi'])
        print(f"    {s:25s}: full_IC={ic_full:.4f}  conditional_IC={ic_cond:.4f}  "
              f"(n={neutral.sum()})")


1. Coincidence rate (P(both fire | either fires)):


,sig_bad_leader,sig_qw_any,sig_vwap_signed,sig_vwap_strength,sig_vwap_csrank
sig_bad_leader,1.000,0.045,0.060,0.060,0.030
sig_qw_any,0.045,1.000,0.151,0.151,0.507
sig_vwap_signed,0.060,0.151,1.000,1.000,0.116
sig_vwap_strength,0.060,0.151,1.000,1.000,0.116
sig_vwap_csrank,0.030,0.507,0.116,0.116,1.000



2. IC correlation (signal value correlation):


,sig_bad_leader,sig_qw_any,sig_vwap_signed,sig_vwap_strength,sig_vwap_csrank
sig_bad_leader,1.000,0.005,0.016,0.021,-0.030
sig_qw_any,0.005,1.000,0.011,0.004,-0.004
sig_vwap_signed,0.016,0.011,1.000,-0.003,0.009
sig_vwap_strength,0.021,0.004,-0.003,1.000,-0.004
sig_vwap_csrank,-0.030,-0.004,0.009,-0.004,1.000



3. Conditional IC (unique contribution):
    sig_bad_leader           : full_IC=-0.0051  conditional_IC=0.0288  (n=11261)
    sig_qw_any               : full_IC=-0.0043  conditional_IC=-0.0049  (n=22391)
    sig_vwap_signed          : full_IC=-0.0196  conditional_IC=-0.0499  (n=11139)
    sig_vwap_strength        : full_IC=0.0035  conditional_IC=0.0141  (n=11139)
    sig_vwap_csrank          : full_IC=0.0134  conditional_IC=0.0005  (n=38494)


## Signal Combination

Combine signals into a composite score:
1. **Equal weight**: w_i = 1/n
2. **IC weight**: w_i = IC_i / sum|IC_j|
3. **Shrinkage Markowitz**: (1-lambda)*inv(Sigma)*IC + lambda/n

In [26]:

# Signal combination methods
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]

if not active_cols:
    print("No active signals found")
else:
    print(f"Combining {len(active_cols)} signals: {active_cols}")

    # 1. Equal weight
    w_equal = pd.Series(1.0 / len(active_cols), index=active_cols)

    # 2. IC weight
    ic_vals = {c: compute_event_ic(c_val[c], c_val['copyable_roi']) or 0.0
               for c in active_cols}
    ic_sum = sum(abs(v) for v in ic_vals.values())
    w_ic = pd.Series({c: ic_vals[c] / ic_sum if ic_sum > 0 else 1.0/len(active_cols)
                       for c in active_cols})

    # 3. Shrinkage Markowitz
    w_shrink = compute_optimal_weights(c_val, active_cols, 'copyable_roi', shrinkage=0.5)

    schemes = {
        'equal': w_equal,
        'ic_weighted': w_ic,
        'shrinkage_markowitz': w_shrink,
    }

    for name, w in schemes.items():
        print(f"\n  {name}:")
        for c, wt in w.items():
            print(f"    {c:25s} = {wt:.4f}")

    # Apply composite scores
    for name, w in schemes.items():
        for df_c in [c_train, c_val, c_test]:
            df_c[f'composite_{name}'] = apply_composite_score(df_c, active_cols, w)

    # Compare on validation
    comp_cols = [f'composite_{k}' for k in schemes]
    comp_results = []
    for cc in comp_cols:
        ic_c = compute_event_ic(c_val[cc], c_val['copyable_roi'])
        ir_c = compute_event_ir(c_val[cc], c_val['copyable_roi'], c_val['dt'], freq='D')
        comp_results.append({'composite': cc, 'IC': ic_c, 'IR': ir_c})
    comp_df = pd.DataFrame(comp_results)
    print("\n\nComposite signal quality (validation):")
    display(comp_df.round(4))


Combining 5 signals: ['sig_bad_leader', 'sig_qw_any', 'sig_vwap_signed', 'sig_vwap_strength', 'sig_vwap_csrank']



  equal:
    sig_bad_leader            = 0.2000
    sig_qw_any                = 0.2000
    sig_vwap_signed           = 0.2000
    sig_vwap_strength         = 0.2000
    sig_vwap_csrank           = 0.2000

  ic_weighted:
    sig_bad_leader            = -0.1105
    sig_qw_any                = -0.0948
    sig_vwap_signed           = -0.4276
    sig_vwap_strength         = 0.0757
    sig_vwap_csrank           = 0.2914

  shrinkage_markowitz:
    sig_bad_leader            = -0.1326
    sig_qw_any                = -0.1041
    sig_vwap_signed           = -0.2646
    sig_vwap_strength         = -0.1578
    sig_vwap_csrank           = 0.3410




Composite signal quality (validation):


,composite,IC,IR
0,composite_equal,-0.0062,-0.0585
1,composite_ic_weighted,-0.0014,0.0654
2,composite_shrinkage_markowitz,-0.0053,0.0789


## Strategy Evaluation

When composite_score >= threshold, copy the BUY trade. Grid-search threshold on validation.

In [27]:

# Strategy evaluation: when composite_score >= threshold, copy the trade
# Use Markowitz composite if available, fall back to IC-weighted, then equal

best_composite = 'composite_shrinkage_markowitz'
if best_composite not in c_val.columns or c_val[best_composite].notna().sum() < 10:
    best_composite = 'composite_ic_weighted'
if best_composite not in c_val.columns or c_val[best_composite].notna().sum() < 10:
    best_composite = 'composite_equal'
print(f"Using: {best_composite}")


def evaluate_strategy(df, score_col, threshold):
    fired = df[df[score_col] >= threshold].copy()
    if fired.empty:
        return {
            'threshold': threshold, 'trades': 0,
            'copyable_pnl': 0.0, 'copyable_roi': 0.0,
            'total_pnl': 0.0, 'notional': 0.0,
            'copyable_notional': 0.0, 'firing_rate': 0.0,
        }
    cnot = fired['copyable_notional'].sum()
    return {
        'threshold': threshold,
        'trades': len(fired),
        'copyable_pnl': float(fired['copyable_pnl'].sum()),
        'copyable_roi': float(fired['copyable_pnl'].sum() / cnot) if cnot > 0 else 0.0,
        'total_pnl': float(fired['pnl'].sum()),
        'notional': float(fired['notional'].sum()),
        'copyable_notional': float(cnot),
        'firing_rate': len(fired) / len(df),
    }


# Grid search on validation
thresholds = np.arange(0.0, 1.05, 0.05)
val_results = [evaluate_strategy(c_val, best_composite, t) for t in thresholds]
val_df = pd.DataFrame(val_results)
val_df['pnl_per_trade'] = val_df['copyable_pnl'] / val_df['trades'].clip(lower=1)

print("\nGrid search (validation): top 10 by copyable_pnl")
display(val_df.sort_values('copyable_pnl', ascending=False).head(10).round(2))

# Pick best threshold (max copyable_pnl, min 20 trades)
candidates = val_df[val_df['trades'] >= 20]
if not candidates.empty:
    best_row = candidates.sort_values('copyable_pnl', ascending=False).iloc[0]
else:
    best_row = val_df.sort_values('copyable_pnl', ascending=False).iloc[0]
best_threshold = best_row['threshold']
print(f"\nBest threshold: {best_threshold:.2f}  "
      f"(copyable_pnl=${best_row['copyable_pnl']:,.0f}, "
      f"{best_row['trades']} trades)")


Using: composite_shrinkage_markowitz

Grid search (validation): top 10 by copyable_pnl


,threshold,trades,copyable_pnl,copyable_roi,total_pnl,notional,copyable_notional,firing_rate,pnl_per_trade
20,1.00,48,-240.02,-0.77,-242.60,312.59,310.01,0.0,-5.00
19,0.95,49,-245.02,-0.78,-247.60,317.59,315.01,0.0,-5.00
18,0.90,50,-245.08,-0.78,-247.66,317.65,315.07,0.0,-4.90
17,0.85,52,-248.52,-0.78,-251.10,321.09,318.51,0.0,-4.78
16,0.80,56,-282.41,-0.80,-285.34,355.33,352.40,0.0,-5.04
15,0.75,57,-284.38,-0.80,-287.32,357.31,354.37,0.0,-4.99
11,0.55,70,-300.77,-0.69,-338.07,484.26,433.97,0.0,-4.30
10,0.50,72,-305.30,-0.70,-342.60,488.79,438.50,0.0,-4.24
9,0.45,75,-313.33,-0.70,-353.59,499.79,446.53,0.0,-4.18
13,0.65,62,-318.83,-0.82,-362.06,432.05,388.82,0.0,-5.14



Best threshold: 1.00  (copyable_pnl=$-240, 48.0 trades)


In [28]:

# Test set evaluation
test_result = evaluate_strategy(c_test, best_composite, best_threshold)
all_result = evaluate_strategy(c_test, best_composite, -np.inf)

print("Test set evaluation:")
print(f"  Threshold: {best_threshold:.2f}")
print(f"  Trades fired: {test_result['trades']:,} / {all_result['trades']:,} "
      f"({test_result['firing_rate']:.1%})")
print(f"  Copyable PnL: ${test_result['copyable_pnl']:,.0f}")
print(f"  Copyable ROI: {test_result['copyable_roi']:.4f}")
print(f"  Total PnL: ${test_result['total_pnl']:,.0f}")
print(f"  PnL per trade: ${test_result['copyable_pnl'] / max(test_result['trades'], 1):.2f}")
print()
print("vs. copying all candidate trades:")
print(f"  Copyable PnL (all): ${all_result['copyable_pnl']:,.0f}")
print(f"  Copyable ROI (all): {all_result['copyable_roi']:.4f}")

# Summary across splits
print("\n\n=== Strategy Summary ===")
for label, df_i, th in [('Train', c_train, best_threshold),
                          ('Val', c_val, best_threshold),
                          ('Test', c_test, best_threshold)]:
    r = evaluate_strategy(df_i, best_composite, th)
    ra = evaluate_strategy(df_i, best_composite, -np.inf)
    print(f"  {label:6s}: threshold={th:.2f}  "
          f"trades={r['trades']:>5,}/{len(df_i):>6,}  "
          f"cpnl=${r['copyable_pnl']:>8,.0f}  "
          f"croi={r['copyable_roi']:.4f}  "
          f"(all: cpnl=${ra['copyable_pnl']:>8,.0f}  croi={ra['copyable_roi']:.4f})")


Test set evaluation:
  Threshold: 1.00
  Trades fired: 25 / 61,196 (0.0%)
  Copyable PnL: $-173
  Copyable ROI: -1.0000
  Total PnL: $-173
  PnL per trade: $-6.91

vs. copying all candidate trades:
  Copyable PnL (all): $13,434
  Copyable ROI (all): 0.0720


=== Strategy Summary ===
  Train : threshold=1.00  trades=   71/70,868  cpnl=$    -595  croi=-0.9803  (all: cpnl=$  61,080  croi=0.2193)
  Val   : threshold=1.00  trades=   48/82,959  cpnl=$    -240  croi=-0.7742  (all: cpnl=$   4,926  croi=0.0194)
  Test  : threshold=1.00  trades=   25/61,196  cpnl=$    -173  croi=-1.0000  (all: cpnl=$  13,434  croi=0.0720)


In [29]:

# Leave-one-out signal contribution (validation)
active_cols = [c for c in signal_cols if c in c_val.columns and c_val[c].notna().sum() > 10]

if len(active_cols) >= 2:
    print("Signal contribution (leave-one-out on validation):")
    full_ic = compute_event_ic(c_val[best_composite], c_val['copyable_roi'])

    loo_results = []
    for leave_out in active_cols:
        remaining = [c for c in active_cols if c != leave_out]
        if not remaining:
            continue
        w = compute_optimal_weights(c_val, remaining, 'copyable_roi', shrinkage=0.5)
        c_val[f'composite_loo_{leave_out}'] = apply_composite_score(c_val, remaining, w)
        ic_loo = compute_event_ic(c_val[f'composite_loo_{leave_out}'], c_val['copyable_roi'])
        loo_results.append({'left_out': leave_out, 'IC': ic_loo, 'IC_drop': full_ic - ic_loo})

    loo_df = pd.DataFrame(loo_results).sort_values('IC_drop', ascending=False)
    print(f"  Full composite IC: {full_ic:.4f}")
    display(loo_df.round(4))


Signal contribution (leave-one-out on validation):


  Full composite IC: -0.0053


,left_out,IC,IC_drop
3,sig_vwap_strength,-0.0104,0.0051
0,sig_bad_leader,-0.0095,0.0042
2,sig_vwap_signed,-0.0032,-0.0022
4,sig_vwap_csrank,0.0002,-0.0056
1,sig_qw_any,0.0050,-0.0104


## Save Results

In [30]:

# Persist results
import json
from datetime import datetime, timezone
from pathlib import Path

# Signal ICs on TRAIN only (no test leakage)
signal_ics = {}
for col in active_cols:
    signal_ics[col] = {
        "IC": compute_event_ic(c_train[col], c_train['copyable_roi']),
        "IR": compute_event_ir(c_train[col], c_train['copyable_roi'], c_train['dt'], freq='D'),
        "hit_rate": hit_rate(c_train[col], c_train['copyable_roi']),
    }

output = {
    "stage": 1,
    "type": "experimental_signal_framework",
    "metadata": {
        "run_timestamp": datetime.now(timezone.utc).isoformat(),
        "TEST_MODE": TEST_MODE,
        "n_copy_wallets": len(copy_wallets),
        "n_quality_wallets": len(quality_wallets),
        "n_candidate_trades": len(candidate_trades),
        "best_threshold": float(best_threshold),
        "best_composite": best_composite,
        "signal_windows_min": {
            "bad_leader": BAD_LEADER_WINDOW,
            "quality_wallet": QUALITY_WALLET_WINDOW,
            "vwap": VWAP_WINDOW,
        },
    },
    "signals": {
        col: vals for col, vals in signal_ics.items()
    },
    "weights": {
        "equal": {k: float(v) for k, v in w_equal.items()},
        "ic_weighted": {k: float(v) for k, v in w_ic.items()},
        "shrinkage_markowitz": {k: float(v) for k, v in w_shrink.items()},
    },
    "val_performance": val_df.round(4).to_dict(orient="records"),
    "test_performance": {
        "threshold": best_threshold,
        "copyable_pnl": test_result["copyable_pnl"],
        "copyable_roi": test_result["copyable_roi"],
        "total_pnl": test_result["total_pnl"],
        "trades": test_result["trades"],
        "firing_rate": test_result["firing_rate"],
        "all_trades_copyable_pnl": all_result["copyable_pnl"],
        "all_trades_copyable_roi": all_result["copyable_roi"],
    },
}

out_path = Path("stage1_experimental_result.json")
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"Saved -> {out_path.resolve()}")


Saved -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_experimental_result.json


## TODO: Next Iterations


# TODO: Next Iterations

### Short-term
- [ ] **Full VWAP** - per-trade rolling VWAP (numba-accelerated like _twopass_impl.py)
- [ ] **Aggregated quality wallet volume** - sum all quality wallet volume in window, not just nearest
- [ ] **Top buyer position signal** - track cumulative positions of top N buyers per market
- [ ] **Disagreement signal** - divergence between top buyers (some buying YES, others NO)
- [ ] **Walk-forward cross-validation** - replace single val split

### Medium-term
- [ ] **Deflated Sharpe Ratio** (Bailey et al. 2014) - correct for multiple-signal testing
- [ ] **Non-linear combination** - shallow gradient-boosted ensemble over raw signals
- [ ] **Rolling IC estimation** - re-estimate weights on expanding monthly windows
- [ ] **Calibration layers** - port price_bucket and consensus scores from signal/scorer.py
- [ ] **Execution tape integration** - feed into backtest/execution_tape.py with slippage & latency

### Long-term
- [ ] **Meta-signal from wallet groups** - use polymarket_analysis.copy_groups as signal input
- [ ] **Regime detection** - adjust signal weights per market regime
- [ ] **Full portfolio backtest** - multi-wallet, multi-market with Kelly sizing & risk limits
